# Which NanoString samples are worth sending for RNA-seq?
### Notebook `05_rnaseq_sample_selector` (Python; self-contained: JDM workbook + clinical database CSV)

**The idea.** A higher interferon score usually means more active disease, so the score can rank samples for RNA-seq: high-score samples for the active end, low-score ones for the quiet end.

**The catch you raised.** The NanoString sample date can be far from any clinical visit, and "active disease" is only known at a visit. So for every NanoString sample this notebook finds the **clinical visit closest in date** (same patient), records how many days apart they are, and only trusts the clinical picture when that gap is small.

**What it does.**
1. Scores every JDM sample (geomean28, as in notebook 03).
2. Finds each sample's closest visit by date and reports the gap.
3. Shows how the score-versus-activity relationship weakens as the gap grows, so the gap cut-off is chosen from your data, not guessed.
4. Ranks eligible samples and writes a candidate table: score, tier (high / middle / low), matched visit, gap, activity.

**What it does not do.** It does not look at RNA-seq. Whether a patient has stored RNA or sequenced material is for you to filter afterwards; the table has one row per sample so that is easy.

## Part 1 · Settings
💡 Every choice is here. The cut-offs are set before looking at results.

In [ ]:
import os, re, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

OUT        = "/mnt/filesystems/nanostring_out"
SEARCH     = ["/mnt/filesystems", "/mnt/session_data", "/home/jovyan"]
XLSX_REGEX = r"^JDM IFN scores OADR.*\.xlsx$"
DB_REGEX   = r"^OADR-I-26-007.*clinical.*JDM.*\.csv$"
XLSX = None; DB = None; RAW_SHEET = None          # or give full paths / the exact sheet name

HK    = ["ALAS1", "HPRT1", "TBP", "TUBB"]
IFN28 = ["CXCL10","DDX60","EPSTI1","GBP1","HERC5","HERC6","IFI27","IFI44","IFI44L","IFI6","IFIT1","IFIT2",
         "IFIT3","IFIT5","ISG15","LAMP3","LY6E","MX1","OAS1","OAS2","OAS3","OASL","RSAD2","RTP4","SIGLEC1",
         "SOCS1","SPATS2L","USP18"]
FLOOR = 0.5

# how a patient is read off a code like 015-01-01 (same rule as notebook 03): everything before the first "-"
def patient_of(x): return x.astype(str).str.strip().str.split("-").str[0].str.upper()

# clinical database columns (same names as notebook 03)
DB_NANO_ID    = "nano ID"
VISIT_DATE_COL = None            # None = guessed in Part 3 (printed, so you can check); or the exact column name
ACTIVITY       = "MD_GLOBAL_ACT_ADJ_VAS"          # the activity measure used to judge a "good" match
OTHER_CLIN = {"DAS_total": "DAS_TOTAL_ADJ_SCORE", "MMT8": "MMT_MMT8_ADJUSTED_SCORE", "CK": "Observation Value_CK"}

# gap handling, fixed in advance
GAP_BINS      = [30, 90, 180, 365]        # days; Part 5 reports the score-activity link inside each band
MAX_GAP_DAYS  = 90                        # a sample is ELIGIBLE for selection only if its closest visit is this near
MIN_N_BAND    = 8                         # a band with fewer samples gets no correlation
HIGH_Q, LOW_Q = 2/3, 1/3                  # tiers by score quantile among eligible samples
ONE_PER_PATIENT = True                    # in the final table also mark each patient's best sample

## Part 2 · The JDM samples and their score
🎯 **Goal:** one row per NanoString sample with its geomean28 score and its date. 💡 **Same recipe as notebook 03:** scale each sample so its four housekeeping genes have a geometric mean of 1000, then take the geometric mean of the 28 interferon genes, divided by 10.

In [ ]:
def find_one(pat):
    hits = []
    for d in SEARCH:
        if os.path.isdir(d):
            for root, _, files in os.walk(d):
                hits += [os.path.join(root, f) for f in files if re.search(pat, f, re.I) and not f.startswith("~$")]
    if not hits: raise FileNotFoundError(f"no file matching {pat} under {SEARCH}")
    return sorted(hits)[0]

def to_date(s):
    """Excel serial numbers or ordinary date text -> datetime64; unparseable -> NaT."""
    s = pd.Series(s)
    num = pd.to_numeric(s, errors="coerce")
    out = pd.to_datetime(s, errors="coerce")
    serial = num.between(20000, 80000)
    out[serial] = pd.to_datetime(num[serial], unit="D", origin="1899-12-30")
    return out

XLSX = XLSX or find_one(XLSX_REGEX); DB = DB or find_one(DB_REGEX)
sheets = pd.ExcelFile(XLSX).sheet_names
sh = RAW_SHEET or next(s for s in sheets if re.search("raw", s, re.I) and not re.search("no|non|form|std", s, re.I))
raw = pd.read_excel(XLSX, sheet_name=sh, header=None, dtype=str).dropna(how="all")
genes = set(IFN28 + HK); head = raw.head(15)
h = int(np.argmax([sum(str(x).strip() in genes for x in head.iloc[i]) for i in range(len(head))]))
hdr = [str(x).strip() if pd.notna(x) else f"unnamed_{j}" for j, x in enumerate(head.iloc[h])]
df = raw.iloc[h + 1:].copy(); df.columns = hdr
need = ["study sample", "sample date"]
if [c for c in need if c not in df]: raise KeyError(f"workbook lacks {[c for c in need if c not in df]}; found {list(df.columns)[:15]}")
miss = [g for g in IFN28 + HK if g not in df]
if miss: raise KeyError(f"genes missing: {miss}")
for g in IFN28 + HK: df[g] = pd.to_numeric(df[g], errors="coerce")
df = df[df["study sample"].notna()].copy()

c = df[IFN28 + HK].clip(lower=FLOOR)
hkg = np.exp(np.log(c[HK]).mean(axis=1))
norm = c.div(hkg, axis=0) * 1000
df["score"] = np.exp(np.log(norm[IFN28]).mean(axis=1)) / 10
df["patient"] = patient_of(df["study sample"])
df["sample_date"] = to_date(df["sample date"])
df = df[df["score"].notna()].reset_index(drop=True)
print(f"workbook: {os.path.basename(XLSX)} | sheet: {sh} | header row {h}")
print(f"samples scored: {len(df)} | patients: {df['patient'].nunique()} | samples with a usable date: {df['sample_date'].notna().sum()}")
print(f"sample dates: {df['sample_date'].min():%Y-%m-%d} to {df['sample_date'].max():%Y-%m-%d}")

## Part 3 · The clinical visits
🎯 **Goal:** every clinical row with a patient and a visit date. 💡 **How the visit date column is found:** if `VISIT_DATE_COL` is `None`, the notebook lists every column whose name contains "date" or "visit" with how many values parse as dates, and takes the best one. **Look at that list**: if it picked the wrong column, set `VISIT_DATE_COL` and rerun from here.

In [ ]:
db = pd.read_csv(DB, dtype=str, low_memory=False)
if DB_NANO_ID not in db: raise KeyError(f"database has no '{DB_NANO_ID}' column; found {list(db.columns)[:20]}")
db["patient"] = patient_of(db[DB_NANO_ID].fillna(""))
db.loc[db[DB_NANO_ID].isna() | (db[DB_NANO_ID].str.strip() == ""), "patient"] = np.nan

NOT_A_VISIT = r"birth|dob|born|death|deceas|diagnos|onset|enrol"      # dates that are never the visit date
cand = [col for col in db.columns if re.search("date|visit", col, re.I)]
rows = []
for col in cand:
    d = to_date(db[col]); rows.append(dict(column=col, parsed=int(d.notna().sum()), of=int(db[col].notna().sum()),
                                           distinct=int(d.nunique()), earliest=d.min(), latest=d.max(),
                                           skipped=bool(re.search(NOT_A_VISIT, col, re.I))))
cand_tab = pd.DataFrame(rows)
cand_tab["says_visit"] = cand_tab["column"].str.contains("visit", case=False)
cand_tab = cand_tab.sort_values(["skipped", "says_visit", "parsed", "distinct"], ascending=[True, False, False, False])
print("columns that look like dates (best guess first; 'skipped' = birth/death/diagnosis-type dates):")
print(cand_tab.drop(columns="says_visit").to_string(index=False))
col = VISIT_DATE_COL or (cand_tab.iloc[0]["column"] if len(cand_tab) else None)
if col is None: raise KeyError("no date-like column in the database; set VISIT_DATE_COL")
print("\nvisit date column used:", col)
db["visit_date"] = to_date(db[col])
if db["visit_date"].nunique() < 5: raise ValueError(f"'{col}' has fewer than 5 distinct dates; set VISIT_DATE_COL to the real visit date")

clin = {"activity": ACTIVITY, **OTHER_CLIN}
for k, v in clin.items(): db[k] = pd.to_numeric(db[v], errors="coerce") if v in db else np.nan
if ACTIVITY not in db: print(f"WARNING: activity column '{ACTIVITY}' not in the database")
visits = db[db["patient"].notna() & db["visit_date"].notna()][["patient", "visit_date", DB_NANO_ID, *clin]].copy()
print(f"\nclinical rows with patient + visit date: {len(visits)} | patients: {visits['patient'].nunique()}")
print(f"patients scored but absent from the clinical database: {len(set(df['patient']) - set(visits['patient']))} of {df['patient'].nunique()}")

## Part 4 · Closest visit for every sample
🎯 **Goal:** for each NanoString sample, the clinical row of the same patient whose visit date is nearest the sample date. `gap_days` is visit minus sample (negative = the visit came *before* the sample). Ties go to the earlier visit.
💡 **Check built in:** notebook 03 matched on the database `nano ID` instead of on date. Where both exist, the notebook says how often the two methods land on the same visit.

In [ ]:
def closest(sample_row):
    v = visits[visits["patient"] == sample_row["patient"]]
    if v.empty or pd.isna(sample_row["sample_date"]): return pd.Series(dtype=object)
    g = (v["visit_date"] - sample_row["sample_date"]).dt.days
    order = np.lexsort((v["visit_date"].values, g.abs().values))     # smallest |gap|, then earlier visit
    j = v.index[order[0]]
    return pd.Series({"gap_days": g[j], "visit_date": v.loc[j, "visit_date"], "visit_nano_id": v.loc[j, DB_NANO_ID],
                      **{k: v.loc[j, k] for k in clin}})

m = df.apply(closest, axis=1)
S = pd.concat([df[["study sample", "patient", "sample_date", "score"]], m], axis=1)
S["abs_gap"] = S["gap_days"].abs()
S["nano_id_match"] = S["study sample"].astype(str).str.strip().str.upper() == S["visit_nano_id"].astype(str).str.strip().str.upper()

has = S["abs_gap"].notna()
if has.any() and S["abs_gap"].median() > 3 * 365:
    raise ValueError(f"median gap is {S['abs_gap'].median()/365:.1f} years: the visit date column ('{col}') is almost certainly the wrong one; "
                     "set VISIT_DATE_COL in Part 1 and rerun from Part 3")
print(f"samples with a closest visit: {has.sum()} of {len(S)}  (no visit found: {(~has).sum()})")
print("\ndays between the sample and its closest visit:")
print(S["abs_gap"].describe(percentiles=[.25, .5, .75, .9]).round(0).to_string())
for t in [0, 7, 30, 90, 180, 365]:
    print(f"  within {t:>3} days: {(S['abs_gap'] <= t).sum():>3}  ({100*(S['abs_gap'] <= t).sum()/max(has.sum(),1):.0f}%)")
print(f"\nclosest-by-date visit is the same visit as the database 'nano ID' match: {S.loc[has,'nano_id_match'].mean()*100:.0f}% of samples")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(S["gap_days"].dropna(), bins=40, color="#4C78A8"); ax[0].axvline(0, color="k", lw=.6)
ax[0].set_xlabel("days: visit minus sample (negative = visit earlier)"); ax[0].set_title("distance to the closest visit")
ax[1].hist(S["abs_gap"].dropna().clip(upper=1000), bins=40, color="#76B7B2"); ax[1].axvline(MAX_GAP_DAYS, color="r", lw=1)
ax[1].set_xlabel("absolute gap, days (clipped at 1000)"); ax[1].set_title(f"red = MAX_GAP_DAYS ({MAX_GAP_DAYS})")
plt.tight_layout(); plt.show()

## Part 5 · How much does the gap matter?
🎯 **Goal:** Spearman correlation of score with the activity measure inside bands of gap size. 💡 **Reading it:** if the correlation is clear when the gap is small and fades when it is large, the score is a real activity marker *and* distant visits are not to be trusted, so `MAX_GAP_DAYS` should sit where the correlation fades. If it is flat everywhere, the gap is not what limits you. Bands with fewer than `MIN_N_BAND` samples show no correlation. **One row per patient is not enforced here** (repeat samples are correlated), so treat the numbers as a guide to the cut-off, not as a test.

In [ ]:
def rho(a, b):
    x = pd.concat([a.reset_index(drop=True), b.reset_index(drop=True)], axis=1).dropna()
    return np.nan if len(x) < 3 else x.rank().corr().iloc[0, 1]

edges = [0] + GAP_BINS + [np.inf]; rows = []
for lo, hi in zip(edges[:-1], edges[1:]):
    inband = S[(S["abs_gap"] >= lo) & (S["abs_gap"] <= hi) if lo == 0 else (S["abs_gap"] > lo) & (S["abs_gap"] <= hi)]
    ok = inband[["score", "activity"]].dropna()
    rows.append(dict(band=f"{int(lo)}-{'inf' if hi==np.inf else int(hi)} days", samples=len(inband), with_activity=len(ok),
                     rho_score_vs_activity=(rho(ok["score"], ok["activity"]) if len(ok) >= MIN_N_BAND else np.nan)))
cum = []
for t in GAP_BINS:
    ok = S[S["abs_gap"] <= t][["score", "activity"]].dropna()
    cum.append(dict(band=f"<= {t} days (cumulative)", samples=int((S['abs_gap'] <= t).sum()), with_activity=len(ok),
                    rho_score_vs_activity=(rho(ok["score"], ok["activity"]) if len(ok) >= MIN_N_BAND else np.nan)))
gap_tab = pd.DataFrame(rows + cum).round(2)
print(gap_tab.to_string(index=False))

ok = S.dropna(subset=["score", "activity", "abs_gap"])
if len(ok):
    plt.figure(figsize=(6, 4)); sc = plt.scatter(ok["activity"], ok["score"], c=np.log10(ok["abs_gap"] + 1), cmap="viridis_r", s=18)
    plt.yscale("log"); plt.colorbar(sc, label="log10(days from visit + 1)"); plt.xlabel(ACTIVITY); plt.ylabel("geomean28 score")
    plt.title("score vs activity; dark = near visit"); plt.tight_layout(); plt.show()

## Part 6 · The candidate table
🎯 **Goal:** rank the samples that can be trusted and label them. 💡 **Rules, in order:** (1) eligible = closest visit within `MAX_GAP_DAYS`; (2) tier by score quantile among eligible samples only: high (top third), low (bottom third), middle; (3) if `ONE_PER_PATIENT`, `best_for_patient` marks each patient's highest-scoring eligible sample (ties go to the earlier date). Samples that are not eligible stay in the table with the reason, so nothing disappears silently.

In [ ]:
S["eligible"] = S["abs_gap"] <= MAX_GAP_DAYS
S["why_not"] = np.where(S["abs_gap"].isna(), "no clinical visit found / no sample date",
               np.where(~S["eligible"], f"closest visit more than {MAX_GAP_DAYS} days away", ""))
E = S[S["eligible"]]
hi_cut, lo_cut = E["score"].quantile(HIGH_Q), E["score"].quantile(LOW_Q)
S["tier"] = np.where(~S["eligible"], "", np.where(S["score"] >= hi_cut, "high", np.where(S["score"] <= lo_cut, "low", "middle")))
S["rank_by_score"] = S["score"].where(S["eligible"]).rank(ascending=False, method="first")
if ONE_PER_PATIENT:
    best = (S[S["eligible"]].sort_values(["patient", "score", "sample_date"], ascending=[True, False, True])
              .drop_duplicates("patient").index)
    S["best_for_patient"] = S.index.isin(best)

print(f"eligible samples: {int(S['eligible'].sum())} of {len(S)}  from {S.loc[S['eligible'],'patient'].nunique()} patients")
print(f"tier cut-offs among eligible: low <= {lo_cut:.0f} | high >= {hi_cut:.0f}")
print(S.loc[S["eligible"], "tier"].value_counts().to_string())
print("\nnot eligible, by reason:"); print(S.loc[~S["eligible"], "why_not"].value_counts().to_string())
cols = ["study sample", "patient", "sample_date", "score", "tier", "rank_by_score", "best_for_patient", "visit_date", "gap_days", "activity",
        "DAS_total", "MMT8", "CK", "eligible", "why_not", "nano_id_match"]
final = S[[c for c in cols if c in S]].sort_values(["eligible", "rank_by_score"], ascending=[False, True])
print("\ntop of the ranking:"); print(final.head(10).to_string(index=False))

## Part 7 · Save
`rnaseq_candidates.csv` has patient codes and dates, so keep it on Lifebit. `gap_vs_activity.csv` has no identifiers.

In [ ]:
os.makedirs(OUT, exist_ok=True)
final.to_csv(os.path.join(OUT, "rnaseq_candidates.csv"), index=False)
gap_tab.to_csv(os.path.join(OUT, "gap_vs_activity.csv"), index=False)
print("written to", OUT)

## How to use the result

* Read **Part 5** first. It tells you whether a high score on its own is trustworthy and how far a visit can be from the sample before the clinical picture stops being useful. If the cut-off it suggests differs from `MAX_GAP_DAYS`, change the setting and rerun Parts 6 and 7.
* For RNA-seq, filter `rnaseq_candidates.csv` to `eligible` and `best_for_patient`, then to patients with material, then take the `high` and `low` tiers.
* Samples marked `not eligible` are not bad samples; their clinical state is unknown at that date. They can still be scored, just not selected on activity.
* `nano_id_match` below 100% in Part 4 means the date-closest visit is a different row than notebook 03 used. That is expected for patients with several visits, and it is the point of this notebook.